# ASSET-9 stage 2: the scorer on the frozen files

Scorer `0.2.1-stage2`, script `asset9-stage2 2026-10-16`. Colab **CPU** runtime. No API call, no GPU, nothing billed.

| Cell | What | Writes to Drive |
|---|---|---|
| 0 | Drive, repo, one PASS or FAIL line per input | nothing |
| 1 | 431 unit tests (394 of stage 1, 37 on the frozen files) | nothing |
| 2 | audit with `--require-full` and the expected scored sizes | `scored/asset9_stage2/audit/` |
| 3 | forced choice through the score command, made-up answers | `scored/asset9_stage2/fc_dryrun/` |
| 4 | the score command on the pilot caches, counts only | `scored/asset9_stage2/pilot_test/` |
| 5 | the proofreading sheet (send only the .xlsx) | `human/ASSET-9/proofread/` |
| 6 | later: read the filled sheet back | `human/ASSET-9/proofread/read/` |

Printouts give ids and counts only. Files with text stay in Drive.

In [ ]:
# Cell 0: Drive, repo, and one PASS or FAIL line per input. Run first in every session.
import hashlib, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")

D = Path("/content/drive/MyDrive/BanglaIndoorVQA")      # Drive project folder
R = Path("/content/banglaindoorvqa-main")               # repo clone
TOKEN = userdata.get("GITHUB_TOKEN")
if not R.exists():
    subprocess.run(["git", "clone", "-q",
                    f"https://{TOKEN}@github.com/blkeyd/banglaindoorvqa-main.git", str(R)], check=True)
subprocess.run(["git", "-C", str(R), "pull", "-q"], check=True)
subprocess.run(["git", "-C", str(R), "fetch", "-q", "--tags"], check=True)
os.chdir(R)
HEAD = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def run(args):
    """Run a repo script; print its output (ids and counts only) and the exit code."""
    r = subprocess.run([sys.executable] + args, cwd=R, capture_output=True, text=True)
    print(r.stdout, end="")
    if r.stderr.strip():
        print(r.stderr, end="")
    print("exit code", r.returncode)
    return r.returncode

EXPECT = {
    R / "lexicon/lexicon.csv":       "73b6bc48f1bda27863fbad69e26830c5f484d50702543b66297679bc07bc7448",
    R / "lexicon/closed_vocab.csv":  "de36f2905e43784704f8765c9886da1e9a9524d295ff10fac6c75b21f50e4b90",
    R / "probes/excluded_items.csv": "8c51763fad6e3c48f0ae1e137573d00e1d0e79274bf959d9bb8e0bca96b37cdb",
    D / "probes/fc_options.jsonl":   "5288ba63033083b10beab8eb5949f3f6a2ab3968a4baf8eb8bb2e1b0fdcf2e96",
    D / "probes/P-CORE-CLOSED.json": "ea242d5b8adeebc96a0a67e43aa84da6cbbe89c4d8ac8ec39fad5e85bde4ce2d",
    D / "probes/R-TEST.json":        "30cd10956e56e8c1d2b04db2abd8659674bd824c31f2c38a75fce4a349a3029d",
    D / "probes/P-NAME.json":        "fc4bfb1dbda1f6f3a554e196dc049c90a9a72af4376170420f50b5decd7e9035",
    D / "probes/PILOT.json":         "4d9e4cc0eaa839bfaec59c881ce520c95779743958a0766160655fa9fbc2b587",
    R / "src/normalise.py":          "488a644d30d130586aa61737ae271a3ed2ddbefb6990454b3542ac09cccc231a",
    R / "src/fc_options.py":         "872dfe0e396f732d319a8f9f0696fc939b5bf946a2850a3dbb51612c9b31d684",
    R / "tests/test_scoring.py":     "1882189d793271e0bb1296a08ed2f6b6ec6e2bf49236515a85084aa163c896ad",
}
results = []
def check(name, ok, info=""):
    results.append(bool(ok))
    print("PASS" if ok else "FAIL", name, info)

print("repo HEAD", HEAD[:12], "| Python", sys.version.split()[0])
for p, want in EXPECT.items():
    if not p.exists():
        check(p.name, False, "missing: %s" % p)
        continue
    got = sha(p)
    check(p.name, got == want, "sha256 %s%s" % (got[:12], "" if got == want else ", expected %s" % want[:12]))
check("lexicon/ equals the tag gate-A",
      subprocess.run(["git", "diff", "--quiet", "gate-A", "--", "lexicon/"]).returncode == 0)
check("src/scoring.py is version 0.2.1-stage2",
      'SCORER_VERSION = "0.2.1-stage2"' in (R / "src/scoring.py").read_text(encoding="utf-8"))
check("scripts/asset9_scorer.py is asset9-stage2 2026-10-16",
      'SCRIPT_VERSION = "asset9-stage2 2026-10-16"' in (R / "scripts/asset9_scorer.py").read_text(encoding="utf-8"))
check("tests/test_real_files.py present", (R / "tests/test_real_files.py").exists())
check("tests/fixtures present", (R / "tests/fixtures/lexicon_FIXTURE.csv").exists()
      and (R / "tests/fixtures/closed_vocab_FIXTURE.csv").exists())
CACHES = [str(D / "outputs/pilot" / m / "cache.jsonl") for m in ("gemini", "qwen", "gemma")]
for c in CACHES:
    n = sum(1 for line in open(c, encoding="utf-8") if line.strip()) if os.path.exists(c) else 0
    check(c.split("outputs/")[1], n > 0, "%d lines" % n)
try:
    import openpyxl
    check("openpyxl", True, openpyxl.__version__)
except ImportError:
    check("openpyxl", False, "run: pip install -q openpyxl")
print("CELL 0: %d PASS, %d FAIL" % (sum(results), len(results) - sum(results)))
assert all(results), "Fix every FAIL line before going on."

In [ ]:
# Cell 1: unit tests. The A2 test reads the probe files from Drive (BIVQA_PROBES).
env = dict(os.environ, BIVQA_PROBES=str(D / "probes"))
r = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"],
                   cwd=R, env=env, capture_output=True, text=True)
print(r.stderr[-4000:] if r.returncode else "\n".join(r.stderr.strip().splitlines()[-3:]))
print("UNIT TESTS:", "PASS" if r.returncode == 0 else "FAIL")

In [ ]:
# Cell 2: the audit with the frozen files and --require-full (B5 task 1), the 5C agreement
# (A2, task 2), the extra words (task 3), and the scored sizes of PROJECT_STATE.
EXPECTED = {
    "P-CORE-CLOSED.json": {"total": 1109, "obj_name": 252, "object": 249, "count": 247,
                           "scene_id": 148, "obj_property": 213},
    "R-TEST.json": {"total": 1435, "obj_name": 276, "object": 369, "count": 363,
                    "scene_id": 209, "obj_property": 218},
    "P-NAME.json": {"total": 991},
    "PILOT.json": {"total": 97},
}
run(["scripts/asset9_scorer.py", "audit", "--require-full",
     "--probe", str(D / "probes/P-CORE-CLOSED.json"), "--probe", str(D / "probes/R-TEST.json"),
     "--probe", str(D / "probes/P-NAME.json"), "--probe", str(D / "probes/PILOT.json"),
     "--out-dir", str(D / "scored/asset9_stage2/audit"), "--expect", json.dumps(EXPECTED)])

In [ ]:
# Cell 3: forced choice through the score command on the real option file (B5 task 4).
# Made-up answers only (right label, wrong label, right text, other-script label, empty);
# no model output is read. P-NAME.json holds all 492 P-CLUST items.
run(["scripts/asset9_scorer.py", "dryrun-fc",
     "--options-file", str(D / "probes/fc_options.jsonl"),
     "--probe", str(D / "probes/P-NAME.json"),
     "--out-dir", str(D / "scored/asset9_stage2/fc_dryrun")])

In [ ]:
# Cell 4: the score command on the real pilot caches, counts only. The Pilot reading chat
# reads Gates C and D; this cell only shows that real cache lines score without error.
# If the Pilot reading is running Cell 8 now, run this cell again after it ends.
args = ["scripts/asset9_scorer.py", "score", "--probe", str(D / "probes/PILOT.json"),
        "--options-file", str(D / "probes/fc_options.jsonl"),
        "--out", str(D / "scored/asset9_stage2/pilot_test/pilot_scored.jsonl"),
        "--counts-only", "--sample", "10"]
for c in CACHES:
    args += ["--outputs", c]
run(args)

In [ ]:
# Cell 5: the proofreading sheet (addendum B5, F5): every distinct no-image pilot answer
# (L-BB0, L-EE0, three models, excluded items left out), blind to the scorer, plus the
# ending list and the refusal phrases. Send ONLY ASSET9_proofread_sheet.xlsx; never the key.
args = ["scripts/asset9_scorer.py", "proofread-sheet", "--probe", str(D / "probes/PILOT.json"),
        "--out-dir", str(D / "human/ASSET-9/proofread")]
for c in CACHES:
    args += ["--outputs", c]
run(args)

In [ ]:
# Cell 6 (LATER, when the sheet comes back): save the returned file as
# human/ASSET-9/proofread/received/ASSET9_proofread_sheet_filled.xlsx, then run this cell.
P = D / "human/ASSET-9/proofread"
args = ["scripts/asset9_scorer.py", "proofread-read",
        "--sheet", str(P / "received/ASSET9_proofread_sheet_filled.xlsx"),
        "--key", str(P / "proofread_key.csv"),
        "--probe", str(D / "probes/PILOT.json"), "--out-dir", str(P / "read")]
for c in CACHES:
    args += ["--outputs", c]
run(args)